# Module 2 Assignment: Adding functionalities to your Automatic Code Review Crew

Welcome to Module 2's assignment! In this lab, you will build upon the Automatic Code Review Crew you created in the assignment in Module 1. This time, you will add the new functionalities you learned in this lesson to take your crew's performance to the next level.

## Background
Your first try at the automation tool for code reviewing was a success. You want to go one step further by adding memory, guardrails and execution hooks to ensure you get the best possible results out of your tool.

Here is a visual summary of the structure of your crew, as well as the new elements you will be adding: 
<div style="text-align: center;">
<img src="./images/agents-tasks-diagram.png" width=600>
</div>

## General instructions for grading
- Replace all `None` instances with your own solution.
- You can add new cells to experiment, but these will be omitted by the grader. Only use the provided cells for your solution code.
- Before submitting, make sure all the cells in your lab work correctly.
- **Do not change variable names**: if you modify variable names, the grader won't be able to find your solutions
- **Use the provided configuration**: for grading, please use all provided configurations. Don't change the configuration files or settings. You can experiment after submitting your lab.
- To submit your notebook, save it and then click on the red **Submit Assignment** button at the top right of the page.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

1. [Set up your notebook](#1)
2. [Agents](#2)
3. [Guardrails](#3)
   - [Exercise 1: Write the `security_review_output_guardrail` guardrail](#ex1)
   - [Exercise 2: Write the `review_decision_guardrail` guardrail](#ex2)
4. [Tasks](#4)
   - [Exercise 3: Create the Analyze Code Quality task](#ex3)
   - [Exercise 4: Create the Review Security task](#ex4)
   - [Exercise 5: Create the Review Decision task](#ex5)
5. [Execution Hooks](#5)
   - [Exercise 6: Create a hook to read the PR file](#ex6)
6. [Creating the Crew](#6)
   - [Exercise 7: Define the Crew](#ex7)
   - [Exercise 8: Kickoff the Crew](#ex8)

<a id="1"></a>

## 1 - Set up your notebook

Begin by importing all necessary modules and configure your environment variables to connect to the LLM APIs.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:

`!pip install crewai[tools]=1.3.0`

<a id="1-1"></a>

### 1.1 - Import modules
Run the following cell to import all the modules you will need for this lab.

In [38]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from pydantic import BaseModel
from utils import get_openai_api_key, get_serper_api_key, clean_markdown
from IPython.display import Markdown, display
import yaml
import os
os.environ["CREWAI_TESTING"] = "true"

In [39]:
import unittests
import dill

<a id="1-2"></a>

### 1.2 - Setup the environment variables

Next, set up the environment variables to connect to the APIs, and create the LLM instance you will use for your Agents

In [40]:
# set up the OpenAI model
os.environ["MODEL"] = "gpt-4o-mini"

# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

# set the Serper API key for the WebsiteSearchTool
os.environ["SERPER_API_KEY"] = get_serper_api_key()

<a id="1-3"></a>

### 1.3 - Import configuration files

Since you already defined the agents and tasks in Module 1's assignment, this time you will just load the parameters from a `YAML` file. These files contain `role`, `goal` and `backstory` for Agents, and `description` and `expected_output` for Tasks. This way, you only need to set up the new parameters and functionalities.

Run the next cell to define the configurations.

In [41]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

<a id="2"></a>

## 2 - Agents

You will begin by creating your agents. This time you don't have to write any code, because you will be using exactly the same agents as in the previous assignment. The only difference is that this time the `role`, `goal` and `backstory` for each agent are given to you in the YAML configuration file. 

Run the cell below to create all the agent instances

In [42]:
# Create the tool instances for the security engineer agent
serper_search_tool = SerperDevTool(
    search_url="https://owasp.org",
    base_url=os.getenv("DLAI_SERPER_BASE_URL") or "https://google.serper.dev",
)
scrape_website_tool = ScrapeWebsiteTool()


# create the Senior Developer agent
senior_developer = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["senior_developer"],
    # set verbose
    verbose=True,
)

# create the Security Engineer agent
security_engineer = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["security_engineer"],
    # add the website search tools (you need to unpack the list)
    tools=[serper_search_tool, scrape_website_tool],
    # set verbose
    verbose=True,
)

# create the Tech Lead agent
tech_lead = Agent(
    # load role, goal, and backstory from the YAML configuration
    config=agents_config["tech_lead"],
    # set verbose
    verbose=True,
)

<a id="3"></a>

## 3 - Guardrails
You need to guarantee security reviews have proper structure and standardized risk levels to prevent vulnerabilities from being misclassified or overlooked. Guardrails provide this essential validation!

You will define two guardrails:


| Name             | Functionality      | Task          |
|------------------|--------------------|---------------|
| `security_review_output_guardrail`| Ensures the security risks are within the specified categories| `review_security`|
| `review_decision_guardrail` | 	Ensures output includes an actionable decision (approve, etc.) | `make_review_decision`|

Each guardrail must return a tuple with a `bool` of whether the checks passed or failed, and a message (optional). If you want to know more details about guardrails in CrewAI, please check the [docs](https://docs.crewai.com/en/concepts/tasks#task-guardrails).

<a id="ex1"></a>

### Exercise 1: Write the `security_review_output_guardrail` guardrail

Complete the cell below to define the function to create the guardrail for the security review output. This guardrail needs to validate: 
- The `risk_level` for each vulnerability is one of the three accepted categories: `low`, `medium` or `high`.
- The `highest_risk` actually matches the highest value in `risk_level`.

Some of the structure is already given to you; you only need to fill in the `None` placeholders.

In [43]:
# GRADED CELL: Exercise 1

import json

def security_review_output_guardrail(output):
    
    # get the (JSON) output from the TaskOutput object
    try: 
        if type(output) == dict:
            json_output = output
        elif output.json_dict is not None:
            json_output = output.json_dict
        else:
            # when a Task has guardrails, CrewAI doesn't parse `raw` into
            # `json_dict` before the first guardrail call, so fall back to
            # parsing the raw LLM text ourselves.
            json_output = json.loads(output.raw)
    except Exception as e:
        return (False, ("Error retrieving the `json_dict` argument: "
                        f"\n{str(e)}\n"
                        "Make sure you set the output_json parameter in the Task."
                        )
                )

    # define risk levels
    valid_risk_levels = ['low', 'medium', 'high']
    
    # Check if security_vulnerabilities key exists
    if 'security_vulnerabilities' not in json_output:
        return (False, f"Missing 'security_vulnerabilities' key in output. Got keys: {list(json_output.keys())}")
    
    # validate that each of the risk levels has a valid value
    for vuln in json_output['security_vulnerabilities']:
        # validate the risk level
        if vuln['risk_level'].lower() not in valid_risk_levels: 
            error_message = f"Invalid risk level: {vuln['risk_level']}"
            return (False, error_message) 
    
    ### START CODE HERE ###
    # validate that the highest risk level matches the highest risk level in the vulnerabilities
    
    # if the highest risk level is not valid risk level, return an error message
    if json_output["highest_risk"].lower() not in valid_risk_levels:
        error_message = f"Invalid highest_risk level: {json_output['highest_risk']}"
        return (False, error_message)
    
    # if it is one of the valid risk levels, then check if it matches the highest 
    # risk level in the vulnerabilities
    else:
        # get all risk_level values
        risk_levels = [vuln['risk_level'].lower() for vuln in json_output['security_vulnerabilities']] 
        
        # if "high" in risk levels, then highest risk level should be high
        if "high" in risk_levels: 
            if json_output["highest_risk"].lower() != "high":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
            
        # if high is not present and medium is in risk levels, then highest risk level should be medium
        elif "medium" in risk_levels: 
            if json_output["highest_risk"].lower() != "medium":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
            
        # if high and medium are not present, then lowest risk level should be low
        elif "low" in risk_levels: 
            if json_output["highest_risk"].lower() != "low":
                error_message = "Highest risk level does not match the highest risk level in the vulnerabilities." 
                return (False, error_message)
    
    ### END CODE HERE ###

    return (True, output.json_dict if type(output) != dict and output.json_dict is not None else json_output)

Try the guardrail with an invalid JSON dictionary, where the highest risk is not correctly identified.

In [44]:
# create the output json for testing
invalid_json = {"highest_risk": "medium", 
                        "security_vulnerabilities": [{"risk_level": "high"}, 
                                                     {"risk_level": "medium"}]}
# test the guardrail
security_review_output_guardrail(invalid_json)

(False,
 'Highest risk level does not match the highest risk level in the vulnerabilities.')

##### **Expected output**:
```
(False,
 'Highest risk level does not match the highest risk level in the vulnerabilities.')
```

In [45]:
# test the guardrail
unittests.test_security_review_output_guardrail(security_review_output_guardrail)

 All tests passed!



<a id="ex2"></a>

### Exercise 2: Write the `review_decision_guardrail` guardrail

Complete the cell below to define the function to create the guardrail for the review decision output. This guardrail needs to make sure the output includes one of the required decision values: "approve", "request changes" or "escalate".

Some of the structure is already given to you; you only need to fill in the `None` placeholders.

In [46]:
# GRADED CELL: Exercise 2

def review_decision_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        "Make sure you set the raw parameter in the Task."
                        )
                )

    # define the keywords to check for in the output
    keywords = ["approve", "request changes", "escalate"]
    
    ### START CODE HERE ###

    # check if any of the keywords are present in the output
    if not any(keyword in output.lower() for keyword in keywords):
        # write a suitable message to the console
        error_message = "Output does not include one of the valid actionable decisions."
        return (False, error_message)

    # if all checks pass, return True, and the output
    return (True, output)

    ### END CODE HERE ###

Try the guardrail with an output missing the possible decision values.

In [47]:
# create the raw output for testing
input = 'Final decision: Elevate to human'

# test the guardrail
review_decision_guardrail(input)

(False, 'Output does not include one of the valid actionable decisions.')

##### **Expected output**:
In this case, the message will vary depending on what you chose
```
(False, 'Output does not include one of the valid actionable decisions.')
```

In [48]:
# test the guardrail
unittests.test_review_decision_guardrail(review_decision_guardrail)

 All tests passed!



<a id="4"></a>

## 4 - Tasks
Now that you have defined the guardrails and execution hooks, you are ready to define the tasks. You will create the three original tasks. For each one, you will load the `description`, `expected_output` and `name` from the yaml file, but you will need to add the guardrails and output types. The difference file (PR) contents will be set in the `file_content` key of the inputs to the crew. 

<a id="ex3"></a>

### Exercise 3: Create the Analyze Code Quality task
Start by defining the Analyze Code Quality task. The structure of the JSON output is defined for you. It includes the following keys:
* `critical_issues`: a list of issues that need fixing
* `minor_issues`: a list of suggested improvements
* `reasoning`: text with the explanation of the reasoning

You should:
1. Define the Task:
    * Use the `config` parameter to load the task configuration from the YAML file
    * Set the `output_json` parameter with the corresponding structure
    * Assign to the corresponding `agent`

In [49]:
# GRADED CELL: Exercise 3

# Define the pydantic model for the code quality analysis output
class CodeQualityJSON(BaseModel):
    critical_issues: list[str]
    minor_issues: list[str]
    reasoning: str

### START CODE HERE ###

# Create the quality analysis task
analyze_code_quality = Task(
    # Load the expected output, and name from the YAML configuration
    config=tasks_config['analyze_code_quality'], 
    # Define the output type as a pydantic model
    output_json=CodeQualityJSON,
    # Define the agent that will perform this task
    agent=senior_developer, 
)

### END CODE HERE ###

In [50]:
# test the analyze_code_quality task
unittests.test_analyze_code_quality(analyze_code_quality)

 All tests passed!



<a id="ex4"></a>

### Exercise 4: Create the Review Security task

1. Define the JSON structure of the Security vulnerabilities. You need the following keys:
    * `description`: string with the description
    * `risk_level`: a string indicating the level
    * `evidence`: a string showing the evidence for the risk

2. Define the JSON structure for the output. You need the following keys:
    * `security_vulnerabilities`: a list of Security vulnerabilities (structure defined before)
    * `blocking`: a boolean indicating if security issues should block approval
    * `highest_risk`: a string with the most severe risk level found
    * `security_recommendations`: a list of strings with specific fixes for identified vulnerabilities

2. Define the Task:
    * Use the `config` parameter to load the task configuration from the YAML file
    * Set the `output_json` parameter
    * Add the guardrail
    * Assign to the corresponding agent

In [51]:
# GRADED CELL: Exercise 4

### START CODE HERE ###

# Define the pydantic model for the security vulnerabilities
class SecurityVulnerability(BaseModel):
    description: str
    risk_level: str
    evidence: str

# Define the pydantic model for the security review output
class ReviewSecurityJSON(BaseModel):
    security_vulnerabilities: list[SecurityVulnerability] 
    blocking: bool
    highest_risk: str
    security_recommendations: list[str]

# Create the security review task
review_security = Task(
    # Load the expected output, agent, and name from the YAML configuration
    config=tasks_config['review_security'], 
    # Define the output type as a pydantic model
    output_json=ReviewSecurityJSON,
    # Add the security_review_output_guardrail guardrail
    guardrails=[security_review_output_guardrail],
    # Define the agent that will perform this task
    agent=security_engineer 
)

### END CODE HERE ###

In [52]:
# test the review_security task
unittests.test_review_security(review_security, SecurityVulnerability, ReviewSecurityJSON)

 All tests passed!



<a id="ex5"></a>

### Exercise 5: Create the Review Decision task

Next, define the Review Decision task. For this task, you will need to set the corresponding `guardrail`. 

1. Define the Task:
    * Use the `config` parameter to load the task configuration from the YAML file
    * Set the `Markdown` parameter, to get the final report in Markdown format
    * Add the guardrail
    * Assign to the corresponding agent

In [53]:
# GRADED CELL: Exercise 5

### START CODE HERE ###

# Create the review decision task
make_review_decision = Task(
    # Load the expected output, agent, and name from the YAML configuration
    config=tasks_config['make_review_decision'], 
    # Set the Markdown parameter to get the final report in Markdown format
    markdown=True,
    # Add the review_decision_guardrail guardrail
    guardrails=[review_decision_guardrail],
    # Set the context with the previous task objects
    context=[analyze_code_quality, review_security], 
    # Define the agent that will perform this task
    agent=tech_lead 
)

### END CODE HERE ###

In [54]:
# test the make_review_decision task
unittests.test_make_review_decision(make_review_decision)

 All tests passed!



<a id="5"></a>

## 5 - Execution Hooks

All your tasks require access to the code difference (PR file), but reading the file isn't really an "agentic" task that requires intelligent decision-making. Instead of having your agents read the file directly, you can create a **before-kickoff hook** that handles this automatically. This hook will read the file and add its content to the crew's inputs before the agents begin their work.

<a id="ex6"></a>

### Exercise 6: Create a hook to read the PR file
Write a function to be used as a hook. This function should:
1. Receive the file path from the `file_path` parameter in the crew's inputs
2. Read the contents of the PR file
3. Add a new key to the `input` dictionary called `file_content` containing the loaded file contents

In [55]:
# GRADED CELL: Exercise 6

# Define the execution hook to read the PR file
def read_file_hook(inputs):
    
    ### START CODE HERE ###
    
    # get the file_path from the inputs
    filename = inputs.get("file_path")
    
    ### END CODE HERE ###

    # if the filename is not provided, raise an error
    if not filename:
        raise ValueError("Missing 'file_path' in inputs")

    # try reading the file
    try:
        with open(filename, "r") as f:
            file_contents = f.read()
    # if there are any issues, raise an error
    except Exception as e:
        raise RuntimeError(f"Failed to read file {filename}: {e}")

    
    ### START CODE HERE ###

    # add the file contents to the inputs
    inputs["file_content"] = file_contents
    
    # return the modified inputs
    return inputs

    ### END CODE HERE ###

In [56]:
# test the read_file_hook
unittests.test_read_file_hook(read_file_hook)

 All tests passed!



<a id="6"></a>

## 6 - Creating the Crew

Now that all the elements are in place, you can define the Crew and kick it off to get the assessment for your PR.

<a id="ex7"></a>

### Exercise 7: Define the Crew
In this step, you will define the crew. You need to set up agents and tasks, just like in the previous module, but this time you will also need to add memory. Adding memory enables agents to remember previously identified security vulnerabilities and coding patterns across multiple pull requests, improving consistency and allowing them to recognize recurring issues without starting from scratch each time.

In [57]:
# GRADED CELL: Exercise 7

### START CODE HERE ###

# Create the code review crew
crew = Crew(
    # add the list of agents
    agents=[senior_developer, security_engineer, tech_lead],
    # add the list of tasks (in order of execution) 
    tasks=[analyze_code_quality, review_security, make_review_decision],
    # add memory to the crew
    memory=True,
    # add the before-kickoff hook to read the PR file
    # you need to pass a list with the hook function from Ex 8
    before_kickoff_callbacks= [read_file_hook]
)

### END CODE HERE ###

In [58]:
# test the crew
unittests.test_crew(crew)

 All tests passed!



<a id="ex8"></a>

### Exercise 8: Kickoff the crew

If all tests from the previous exercises are passed, you are ready to kickoff the crew and save the results in a `dill` file. 

In [59]:
# GRADED CELL: Exercise 8

# define the file path for the PR file
file_path = "files/code_changes.txt" 

### START CODE HERE ###

# kickoff the crew
result = await crew.kickoff_async(
    # add the inputs dictionary with the file path
    inputs={"file_path": file_path}
    )

### END CODE HERE ###

# Save the result to a dill file
with open('result.dill', 'wb') as f:
    dill.dump(result, f)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Task: 1. Review the code changes in the pull request. 2. Identify any style issues, potential bugs, or         │
│  maintainability concerns. 3. For each issue, decide if it's critical (must be fixed) or minor (suggestion).    │
│  PR content:  diff --git a/app/user_auth.py b/app/user_auth.py                                                  │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                            

Tool search_memory executed with result: Found memories:
- (score=0.70) A critical issue identified is the use of string interpolation in SQL queries, which can lead to SQL injection vulnerabilities.
  categories: security, vulnerability, da...
Tool search_memory executed with result: Found memories:
- (score=0.74) The risk level of the identified SQL injection vulnerability is high.
  categories: security, vulnerability, SQL injection
  entities: []
  dates: []
  topics: ['SQL inj...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "critical_issues": [                                                                                         │
│      "The use of string interpolation in SQL queries, which can lead to SQL injection vulnerabilities."         │
│    ],                                                                                                           │
│    "minor_issues": [                                                                                            │
│      "Returning True in logout_user without any checks may not provide enough context, suggesting the addition  │
│  of error handling.",                                                                                           │
│      "Replacing print statements for logging login attempts with proper logging using Python's logging module   │
│  for better maintainability.",                                                                                  │
│      "'time.sleep(1)' could make the system less responsive, recommending to log failed attempts with a         │
│  counter instead."                                                                                              │
│    ],                                                                                                           │
│    "reasoning": "The identified critical issue concerns a serious security vulnerability related to SQL         │
│  injection due to string interpolation in queries, which poses a high risk and must be addressed before         │
│  approval. The minor issues, while important for maintaining code quality and performance, do not constitute a  │
│  security risk and can be addressed at a later time."                                                           │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Task: 1. Review the findings from both the code quality and security analyses. 2. Determine if the pull        │
│  request can be automatically approved based on these findings. 3. If approval is not possible, decide whether  │
│  to request specific changes from the developer or escalate to human review. 4. Provide a clear explanation of  │
│  your decision and any required next steps. PR content:  diff --git a/app/user_auth.py b/app/user_auth.py       │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                     

Tool search_memory executed with result: Found memories:
- (score=0.73) The risk level of the identified SQL injection vulnerability is high.
  categories: security, vulnerability, SQL injection
  entities: []
  dates: []
  topics: ['SQL inj...
Tool search_memory executed with result: Found memories:
- (score=0.71) Recommendations for security fixes include replacing string interpolation in SQL queries with parameterized queries, implementing logging using Python's logging module i...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "security_vulnerabilities": [                                                                                │
│      {                                                                                                          │
│        "description": "The use of string interpolation in SQL queries, which can lead to SQL injection          │
│  vulnerabilities.",                                                                                             │
│        "risk_level": "high",                                                                                    │
│        "evidence": "SQL query: user = db.query(f\"SELECT * FROM users WHERE username = '{username}'\")"         │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "blocking": true,                                                                                            │
│    "highest_risk": "high",                                                                                      │
│    "security_recommendations": [                                                                                │
│      "Replace string interpolation in SQL queries with parameterized queries to prevent SQL injection.",        │
│      "Implement logging using Python's logging module instead of print statements for better                    │
│  maintainability.",                                                                                             │
│      "Add error handling in the logout_user function to provide context when returning True.",                  │
│      "Log failed login attempts with a counter instead of using time.sleep(1) to maintain system                │
│  responsiveness."                                                                                               │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Task: 1. Based on the results from 'analyze_code_quality' and 'review_security', make a final review           │
│  decision. 2. Decide whether to approve the changes, request changes, or escalate to human review. 3.           │
│  Summarize the key findings from both analyses to support your decision. 4. If requesting changes, provide a    │
│  clear list of what must be fixed. 5. If approving, include appropriate comments to accompany the approval. 6.  │
│  If escalating, explain why human review is necessary. 7. Include any additional recommendations or             │
│  observations. PR content:  diff --git a/app/user_auth.py b/app/user_auth.py                                    │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfull

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Final Review Decision                                                                                        │
│                                                                                                                 │
│  **Final Decision**: Request Changes                                                                            │
│                                                                                                                 │
│  ## Summary of Key Findings                                                                                     │
│  1. **Critical Issues**:                                                                                        │
│     - The implementation uses string interpolation in SQL queries, which exposes the application to SQL         │
│  injection vulnerabilities. This is classified as a high-risk security issue that must be addressed             │
│  immediately.                                                                                                   │
│                                                                                                                 │
│  2. **Minor Issues**:                                                                                           │
│     - The function `logout_user` returns `True` without any checks, potentially lacking context for successful  │
│  or failed logout attempts.                                                                                     │
│     - Print statements are used for logging login attempts instead of a dedicated logging mechanism, which can  │
│  lead to maintenance difficulties.                                                                              │
│     - The use of `time.sleep(1)` to prevent timing attacks may reduce system responsiveness; logging failed     │
│  attempts with a counter is recommended instead.                                                                │
│                                                                                                                 │
│  ## Required Changes                                                                                            │
│  - **SQL Injection Prevention**:                                                                                │
│    - Replace string interpolation in SQL queries with parameterized queries to eliminate the risk of SQL        │
│  injection.                                                                                                     │
│                                                                                                                 │
│  - **Error Handling**:                                                                                          │
│    - Add checks to provide context when returning `True` in `logout_user`, ensuring that it handles potential   │
│  errors appropriately.                                                                                          │
│                                                                                                                 │
│  - **Logging Improvements**:                                                                                    │
│    - Substitute `print` statements with the Python logg

Let's check out the final report!

In [60]:
# clean the markdown output, in case the report is wrapped with code fences
clean_report = clean_markdown(result.raw)

# display the final report as markdown
display(Markdown(clean_report))

# Final Review Decision

**Final Decision**: Request Changes

## Summary of Key Findings
1. **Critical Issues**:
   - The implementation uses string interpolation in SQL queries, which exposes the application to SQL injection vulnerabilities. This is classified as a high-risk security issue that must be addressed immediately.

2. **Minor Issues**:
   - The function `logout_user` returns `True` without any checks, potentially lacking context for successful or failed logout attempts.
   - Print statements are used for logging login attempts instead of a dedicated logging mechanism, which can lead to maintenance difficulties.
   - The use of `time.sleep(1)` to prevent timing attacks may reduce system responsiveness; logging failed attempts with a counter is recommended instead.

## Required Changes
- **SQL Injection Prevention**:
  - Replace string interpolation in SQL queries with parameterized queries to eliminate the risk of SQL injection.
  
- **Error Handling**:
  - Add checks to provide context when returning `True` in `logout_user`, ensuring that it handles potential errors appropriately.
  
- **Logging Improvements**:
  - Substitute `print` statements with the Python logging module to enhance maintainability and log effectiveness.
  
- **Enhance Security**:
  - Implement a logging system for failed login attempts using a counter rather than employing `time.sleep(1)`.

## Additional Recommendations
- Review database access patterns to ensure best practices are followed for security and performance.
- Consider establishing a coding standard for error handling and logging across the team to maintain consistency across the codebase.

By addressing these issues, we can enhance both the security and quality of the code before it is merged into the main branch.

You reached the end of the assignment. At this point you are ready to submit for grading.

After submitting and being satisfied with your grade you can take some time to experiment changing the guardrails, or adding new URLs to the Website Reading Tool, or even upload a different pull request file with code differences. Don't be afraid to shake things up!

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 📝 本节课总结 (Module 2 Assignment Recap)

**一句话定位 / One-liner**: This lab turned a basic sequential crew into a **production-grade** one — adding structured outputs, guardrails, hooks, and memory to make agent behavior predictable and auditable.

</div>



<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

### 1️⃣ 结构化输出:`output_json` + Pydantic Model

- `Task(output_json=SomeModel, ...)` 让 Agent 的自由文本回答被**强制**转换成符合 Pydantic schema 的 JSON(`CodeQualityJSON`、`ReviewSecurityJSON`)。
- **Why it matters**: 没有结构化输出,下游代码(guardrail、下一个 task、最终报告)只能对着一坨自然语言猜字段名;有了它,`json_output["highest_risk"]` 这种精确取值才可能。
- 嵌套模型也支持:`ReviewSecurityJSON.security_vulnerabilities: list[SecurityVulnerability]`。

</div>



<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

### 2️⃣ Task Guardrails:跑完不等于对,对了才放行

- Guardrail 是一个**普通函数**,签名固定:接收 `TaskOutput`,返回 `(bool, Any)`。成功 → `(True, 验证/清洗后的结果)`;失败 → `(False, "给 Agent 看的错误信息")`。
- 失败时 CrewAI 会**自动把错误信息喂回给同一个 Agent 重试**(最多 `guardrail_max_retries` 次),不需要你手写重试循环。
- 本节写了两个:
  - `security_review_output_guardrail`:校验 `risk_level` 合法 + `highest_risk` 真的是最高风险
  - `review_decision_guardrail`:确保最终决策文本里有 `approve`/`request changes`/`escalate` 之一
- **本节踩的坑 / Gotcha we actually hit**: 一个 Task **只要挂了 `guardrails`**,CrewAI 在第一次调用 guardrail 之前**不会**把 `raw` 解析进 `json_dict`——`output.json_dict` 第一次拿到的是 `None`。这是 `output_json` + `guardrails` 组合使用时的真实源码行为(`task.py` 里 `elif not self._guardrails and not self._guardrail:` 这一行),不是我们代码写错。解决办法是在 guardrail 里加一层兜底,`json_dict` 为空就自己 `json.loads(output.raw)`。

</div>



<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

### 3️⃣ Execution Hooks:在 crew 跑之前/之后插一脚

| Hook | 触发时机 | 本节用法 |
|---|---|---|
| `before_kickoff_callbacks` | `kickoff()` 一开始、Agent 还没动手之前 | `read_file_hook`:把 `inputs["file_path"]` 对应的 PR 文件内容读出来,塞进 `inputs["file_content"]`,这样 Agent 不用自己"用工具读文件" |
| `after_kickoff_callbacks` | 全部 task 跑完之后(上节课学的) | `save_file_hook`:把最终报告存成本地 Markdown 文件 |

**Why before-kickoff here**: 读文件这种确定性的 I/O 操作,**不是**需要 LLM 推理决策的事,犯不着让 Agent 去调用工具读文件(慢、不稳定、浪费 token)。Hook 是普通 Python 函数,纯代码确定性地完成,比交给 Agent "agentic" 地去做更可靠、更便宜。

</div>



<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

### 4️⃣ Memory:`Crew(memory=True)`

- 开启后,Agent 们能在**同一个 crew 的多次 kickoff 之间**记住之前遇到过的安全漏洞、代码规范问题等模式,而不是每次都从零开始。
- 对应这节课的场景:同一个 Security Engineer agent 下次再审查另一个 PR 时,可能会"记得"上次标记过的某类漏洞模式,判断更一致。
- 这是跟 task 之间 context 传递(上节课学的、同一次 kickoff 内部的信息流动)不同维度的东西——memory 是跨多次 kickoff 的长期记忆。

### 5️⃣ 三个 Task 怎么串起来 (Context Chaining Recap)

`make_review_decision` 显式设置了 `context=[analyze_code_quality, review_security]`,这是我们在 Module 1 学过的"手动指定 context 来源"用法(而不是默认拿前面所有 task 的输出)——Tech Lead 做最终决策时,只看"代码质量分析"和"安全审查"这两份结果,不需要再看别的中间产物。

</div>



<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

### 🐛 本节课实际踩过的两个坑 (Bugs We Actually Hit & Fixed)

1. **`base_url` 传 `None` 校验失败**:`SerperDevTool.base_url` 的类型是严格的 `str`(不是 `str | None`),本地没设 `DLAI_SERPER_BASE_URL` 环境变量时 `os.getenv(...)` 返回 `None`,硬传进去直接 pydantic `ValidationError`。修复:`os.getenv(...) or "https://google.serper.dev"`,拿不到就退回默认地址。
2. **Guardrail 收到 `json_dict=None`**:`review_security` task 同时设了 `output_json` 和 `guardrails`,导致 CrewAI 在第一次调用 guardrail 前跳过了 JSON 解析步骤。修复:guardrail 内部加一层 `json.loads(output.raw)` 兜底。

两个坑的共同套路:**遇到 `None`/`NoneType` 相关的报错时,先搞清楚这个值"本该"从哪里来、为什么没来,而不是急着加 `if ... is None: skip`——很多时候是上游框架的某个条件分支没有触发,得去读一下库的源码确认行为。

</div>



<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走 / Key takeaway**: 这节课的核心是把一个"能跑"的多 agent crew,升级成一个"能信任"的 crew——`output_json` 让输出可被程序解析,`guardrails` 让输出被自动校验+重试,`before/after_kickoff_callbacks` 把确定性的 I/O 从 agentic 推理里剥离出来,`memory` 让 crew 在多次运行间保持一致性。这四件事合在一起,就是从 demo 走向 production 的典型路径。

</div>
